# Process ATUS data

Outcomes:
- Filter and process respondent metadata and export into a csv
- Generate wide-form sequences from time diaries and export into a csv
- Perform clustering on the ATUS sequences
- Export cluster and demographic stratum codes into a csv

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2
from helpers import atus
from mobcalibrate.preprocessing import (
    compute_metrics_for_all_sequences, 
    metrics_cosine_D, 
    fit_weighted_kmedoids_with_restarts, 
    compute_dist_to_medoid_thresholds
)

%config InlineBackend.figure_format = 'retina'

### Configs (START HERE)

In [ ]:
# ======================
# PATHS
# ======================
DATA_DIR = "data/atus"
OUTPUT_DIR = "data/processed"
DY = 24  # data release year

RESP_FILE = f'data/atus/atusresp-03{DY}/atusresp_03{DY}.dat'
CPS_FILE = f'data/atus/atuscps-03{DY}/atuscps_03{DY}.dat'
ACT_FILE = f'data/atus/atusact-03{DY}/atusact_03{DY}.dat'

OUT_PATH_META = f"{OUTPUT_DIR}/atus_meta_test.csv"
OUT_PATH_SEQ = f"{OUTPUT_DIR}/atus_seq_test.csv"

WEIGHT_COL = 'TUFINLWGT'

# ======================
# SAMPLE FILTERS
# ======================
YY = 19  # get data to year 20{TY}
YEARS = list(range(2004, 2020))
CBSA_FILTER = [38060]
AGE_MIN = 18

# ======================
# DEMOGRAPHIC GROUPINGS
# (see ATUS data dictionary to adjust mappings)
# ======================

INCOME_LABELS = ['<35k', '35–75k', '75–100k', '>100k']
INCOME_GROUPS_MAPPING = {
    0: lambda x: x <= 9,
    1: lambda x: 10 <= x <= 13,
    2: lambda x: x == 14,
    3: lambda x: x in [15, 16],
}

AGE_LABELS = ['18–24', '25–44', '45–66', '67+']
AGE_GROUPS_MAPPING = {
    0: range(18, 25),
    1: range(25, 45),
    2: range(45, 67),
    3: range(67, 200),
}

STRATIFICATION_SPEC = {
    "income": {
        "source_col": "INCOME_GROUP",
        "mapping": INCOME_GROUPS_MAPPING,
        "labels": INCOME_LABELS
    },
    "age": {
        "source_col": "AGE",
        "mapping": AGE_GROUPS_MAPPING,
        "labels": AGE_LABELS
    },
}

# ======================
# ALPHABET MAPPINGS
# ======================
TEWHERE_MAP = {
    1: 1, # home
    2: 2, # work
    3: 3, # someone else's home -> other place
    4: 4, # restaurants/bars
    5: 3, # place of worship -> other place
    6: 5, # grocery store
    7: 5, # other store/mall
    8: 6, # school
    9: 7, # outdoors away from home
    31: 3, # gym/health club -> other place
    10: 3, # library -> other place
    11: 3, # other place -> other place
    30: 3, # bank -> other place
    32: 3, # post office -> other place
    89: 3, # unspecified -> other place
    # all other: 3 (modes of transportation/travel -> other place)
} # 7 total letters
TEWHERE_MAP = defaultdict(lambda: 8, TEWHERE_MAP)

TEWHERE_MAP_LABELS = {
    1: "home",
    2: "work",
    3: "other place",
    4: "restaurants/bars",
    5: "stores/mall",
    6: "school",
    7: "outdoors",
    8: "transport"
}

MAP_TO_ALPHA = {
    1: 'A',
    2: 'B',
    3: 'C',
    4: 'C',
    5: 'C',
    6: 'C',
    7: 'C',
    8: 'C' #transport -> other
}
MAP_TO_ALPHA = defaultdict(lambda: 'C', MAP_TO_ALPHA)

# ======================
# PARAMETERS
# ======================
T = 30  # sequence cell width in minutes (e.g. 30 minutes)
K = 4   # number of clusters


In [ ]:
# load and clean respondent data
resp = atus.get_resp(RESP_FILE, CPS_FILE, YEARS, CBSA_FILTER, AGE_MIN)
# load and clean diaries data
diaries = atus.get_diaries(ACT_FILE, resp)
# convert TEWHERE to custom aggregate categories
diaries = atus.map_tewhere(TEWHERE_MAP, diaries)

In [ ]:
# stage diaries for sequencing
staged_diaries = atus.stage_diaries(diaries)
# stage sequences using seq_col as a basis for building sequences
staged_sequence = atus.stage_sequence(staged_diaries, seq_col='WHERE')
# generate sequences
seq_data = atus.sequence(staged_sequence, T)

seq_data

In [ ]:
# CONVERT TO ARRAY OF SEQUENCES WITH ALPHABET STATES
seq_alpha = (seq_data
                .replace({8: 3})    # handle transport as "OTHER"
                .map(lambda x: MAP_TO_ALPHA[x])
                .astype(str)
                .agg(''.join, axis=1)
                .values
            )
seq_alpha

In [ ]:
seq_metrics = compute_metrics_for_all_sequences(
    seq_alpha, 
    home_label="A", 
    work_label="B", 
    all_labels=["A","B","C"]
    )
seq_metrics

In [ ]:
atus_D = metrics_cosine_D(seq_metrics, seq_metrics)

In [ ]:
K = 4
W = resp[WEIGHT_COL].values

cluster_results = fit_weighted_kmedoids_with_restarts(atus_D, W, K)
cluster_results

In [ ]:
compute_dist_to_medoid_thresholds(
    atus_D, 
    cluster_results['medoids'], 
    cluster_results['labels'], 
    percentile=99
)

In [ ]:
atus_meta = atus.build_grouped_meta(
    df=resp,
    base_cols=["TUCASEID", "TUFINLWGT", "GTCBSA"],
    group_specs=STRATIFICATION_SPEC,
    cluster_labels=cluster_results['labels'],
    joint_vars=list(STRATIFICATION_SPEC.keys()),
)
atus_meta

In [ ]:
atus_meta.to_csv(OUT_PATH_META, index=False)
seq_data.to_csv(OUT_PATH_META)